# ДЗ 1. Оптимальная стратегия перемещения таксиста по городу Новинск

In [ ]:
!pip install gym==0.22.0

## Задание 1. Описание марковского процесса

### Среда — город Новинск

Таксист передвигается по сетке **N×N** ячеек (N = 5). На сетке фиксированы **4 места для пассажиров** — **A** (0,0), **B** (4,3), **C** (0,4), **D** (4,0) — и **4 пункта назначения** — **a**, **b**, **c**, **d**, совпадающие по координатам с A, B, C, D.

Пассажир может находиться в любом из четырёх мест и желать доехать до любого из четырёх пунктов назначения.

Таксист **автоматически забирает** пассажира, подъезжая к его месту. Для **высадки** пассажира в нужном месте таксисту необходимо выбрать соответствующее действие (Dropoff).

В начале каждого эпизода случайным образом определяется пара (место пассажира, пункт назначения).

### Формальное определение МППР

**Кортеж:** $\langle S, A, P, R, \gamma \rangle$

### 1. Пространство состояний $S$

$$S = (\text{taxi\_row},\ \text{taxi\_col},\ \text{passenger\_location},\ \text{destination})$$

| Компонента | Значения | Кол-во |
|---|---|---|
| taxi\_row | $\{0, 1, \ldots, N-1\}$ | $N = 5$ |
| taxi\_col | $\{0, 1, \ldots, N-1\}$ | $N = 5$ |
| passenger\_location | $\{A, B, C, D, \text{в\_такси}\}$ | 5 |
| destination | $\{a, b, c, d\}$ | 4 |

$$|S| = 5 \times 5 \times 5 \times 4 = 500$$

### 2. Пространство действий $A$

| Индекс | Действие | Описание |
|--------|----------|----------|
| 0 | South | Движение вниз |
| 1 | North | Движение вверх |
| 2 | East | Движение вправо |
| 3 | West | Движение влево |
| 4 | Pickup | Подобрать пассажира |
| 5 | Dropoff | Высадить пассажира |

$$|A| = 6$$

### 3. Функция переходов $P(s' \mid s, a)$

Переходы **детерминированные**: $P(s' \mid s, a) = 1.0$.

- **Движение (0–3):** такси перемещается на одну ячейку в заданном направлении. Если на пути стена или граница сетки — такси остаётся на месте. Если такси подъезжает к месту пассажира — пассажир автоматически садится в такси.
- **Pickup (4):** если такси на месте пассажира — пассажир садится. Иначе — штраф.
- **Dropoff (5):** если такси на пункте назначения и пассажир в такси — успешная высадка, эпизод завершается. Иначе — штраф.

### 4. Функция вознаграждения $R(s, a, s')$

| Событие | Вознаграждение |
|---------|---------------|
| Каждый шаг | $-1$ |
| Успешная высадка (Dropoff в пункте назначения) | $+20$ |
| Неверный Pickup / Dropoff | $-10$ |

### 5. Коэффициент дисконтирования

$$\gamma \in (0, 1], \quad \text{используем } \gamma = 0.99$$

### 6. Начальное состояние

$s_0$ выбирается равномерно случайно: позиция такси, место пассажира $\in \{A, B, C, D\}$ и пункт назначения $\in \{a, b, c, d\}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import gym

env = gym.make("Taxi-v3")
env.reset()
env.render()

In [ ]:
n_states  = env.observation_space.n
n_actions = env.action_space.n

print(f"состояний: {n_states} действий: {n_actions}")

In [ ]:
print("possible_actions(0) = ", list(range(n_actions)))
print("P[0][0] =", env.P[0][0])

## Задание 2. Value Iteration (семинар 2 + Вариант 1: mdp_taxi_v1)

In [ ]:
try:
    import google.colab
    COLAB = True
except ModuleNotFoundError:
    COLAB = False
    pass

if COLAB:
    !wget https://raw.githubusercontent.com/Tviskaron/mipt/master/2019/RL/02/mdp.py

In [ ]:
transition_probs = {
  's0':{
    'a0': {'s0': 0.5, 's2': 0.5},
    'a1': {'s2': 1}
  },
  's1':{
    'a0': {'s0': 0.7, 's1': 0.1, 's2': 0.2},
    'a1': {'s1': 0.95, 's2': 0.05}
  },
  's2':{
    'a0': {'s0': 0.4, 's2': 0.6},
    'a1': {'s0': 0.3, 's1': 0.3, 's2':0.4}
  }
}
rewards = {
  's1': {'a0': {'s0': +5}},
  's2': {'a1': {'s0': -1}}
}

from mdp import MDP
toy_mdp = MDP(transition_probs, rewards, initial_state='s0')

In [ ]:
state = toy_mdp.reset()
print('initial state =', state)
next_state, reward, done, info = toy_mdp.step('a0')
print(f'next_state ={next_state}, reward = {reward}, done = {done}')

In [ ]:
print("all_states =", toy_mdp.get_all_states())
print("possible_actions('s1') = ", toy_mdp.get_possible_actions('s1'))
print("next_states('s1', 'a0') = ", toy_mdp.get_next_states('s1', 'a0'))
print("reward('s1', 'a0', 's0') = ", toy_mdp.get_reward('s1', 'a0', 's0'))
print("transition_prob('s1', 'a0', 's0') = ",
      toy_mdp.get_transition_prob('s1', 'a0', 's0'))

In [ ]:
for next_state, next_prob in toy_mdp.get_next_states('s0', 'a0').items():
  print(f"{next_state=} , {next_prob=}")

In [ ]:
def get_action_value(mdp, state_values, state, action, gamma):
    """
      Вход: mdp(среда), state_values: dict{str:float}, state: str, action: str
    """
    """ Вычислеям Q(s,a) по формуле выше """
    # вычислеяем оценку состояния
    Q = sum([next_prob * (mdp.get_reward(state, action, next_state) + gamma * state_values[next_state])
              for next_state, next_prob in mdp.get_next_states(state, action).items()])

    return Q

In [ ]:
test_Vs = {s: i for i, s in enumerate(sorted(toy_mdp.get_all_states()))}
assert np.isclose(get_action_value(toy_mdp, test_Vs, 's2', 'a1', 0.9), 0.69)
assert np.isclose(get_action_value(toy_mdp, test_Vs, 's1', 'a0', 0.9), 3.95)

In [ ]:
def get_new_state_value(mdp, state_values, state, gamma):
    """ Считаем следующее V(s) по формуле выше."""
    if mdp.is_terminal(state):
        return 0
    # V =
    V = max([get_action_value(mdp, state_values, state, a, gamma) for a in mdp.get_possible_actions(state)])

    return V

In [ ]:
test_Vs_copy = dict(test_Vs)
assert np.isclose(get_new_state_value(toy_mdp, test_Vs, 's0', 0.9), 1.8)
assert np.isclose(get_new_state_value(toy_mdp, test_Vs, 's2', 0.9), 1.08)
assert np.isclose(get_new_state_value(toy_mdp, {'s0': -1e10, 's1': 0, 's2': -2e10}, 's0', 0.9), -13500000000.0), \
   "Убедитесь, что вы правильно обрабатываете отрицательные значения Q произвольной величины."
assert test_Vs == test_Vs_copy, "Убедитесь, что вы не изменяете state_values в функции get_new_state_value"

In [ ]:
def value_iteration(mdp, state_values=None,
          gamma = 0.9, num_iter = 1000, min_difference = 1e-5):
    """ выполняет num_iter шагов итерации по значениям"""
    # инициализируем V(s)
    state_values = state_values or \
      {s : 0 for s in mdp.get_all_states()}

    for i in range(num_iter):
        # Вычисляем новые полезности состояний,
        # используя функции, определенные выше.
        # Должен получиться словарь {s: new_V(s)}
        new_state_values = {s: get_new_state_value(mdp, state_values, s, gamma) for s in mdp.get_all_states()}


        assert isinstance(new_state_values, dict)

        # Считаем разницу
        diff = max(abs(new_state_values[s] - state_values[s]) for s in mdp.get_all_states())

        print("iter %4i | diff: %6.5f | V(start): %.3f "%
          (i, diff, new_state_values[mdp._initial_state]))

        state_values = new_state_values
        if diff < min_difference:
            print("Принято! Алгоритм сходится!")
            break

    return state_values

In [ ]:
state_values_toy = value_iteration(toy_mdp,
        num_iter = 100, min_difference = 0.001)

In [ ]:
print("Final state values:", state_values_toy)

assert abs(state_values_toy['s0'] - 3.781) < 0.01
assert abs(state_values_toy['s1'] - 7.294) < 0.01
assert abs(state_values_toy['s2'] - 4.202) < 0.01

In [ ]:
def get_optimal_action(mdp, state_values, state,
                       gamma=0.9):
    """ Находим оптимальное действие, используя формулу выше. """
    if mdp.is_terminal(state): return None

    actions = mdp.get_possible_actions(state)
    # выбираем лучшее действие
    i = np.argmax([ get_action_value(mdp, state_values, state, a, gamma) for a in actions])

    return actions[i]

In [ ]:
assert get_optimal_action(toy_mdp, state_values_toy, 's0', 0.9) == 'a1'
assert get_optimal_action(toy_mdp, state_values_toy, 's1', 0.9) == 'a0'
assert get_optimal_action(toy_mdp, state_values_toy, 's2', 0.9) == 'a1'

assert get_optimal_action(toy_mdp, {'s0': -1e10, 's1': 0, 's2': -2e10}, 's0', 0.9) == 'a0', \
    "Убедитесь, что вы правильно обрабатываете отрицательные значения Q произвольной величины."
assert get_optimal_action(toy_mdp, {'s0': -2e10, 's1': 0, 's2': -1e10}, 's0', 0.9) == 'a1', \
    "Убедитесь, что вы правильно обрабатываете отрицательные значения Q произвольной величины."

In [ ]:
s = toy_mdp.reset()
rewards = []
for _ in range(10000):
    s, r, done, _ = toy_mdp.step(get_optimal_action(toy_mdp, state_values_toy, s, 0.9))
    rewards.append(r)

print("average reward: ", np.mean(rewards))

assert(0.40 < np.mean(rewards) < 0.55)

In [ ]:
class TaxiMDPWrapper:
    def __init__(self, env_name='Taxi-v3'):
        self.env = gym.make(env_name)
        self.states = list(range(self.env.observation_space.n))
        self.actions = list(range(self.env.action_space.n))
        self.transitions = self.env.P
        self._initial_state = self.env.reset()

    def get_all_states(self):
        return [state for state in self.states]

    def get_possible_actions(self, state):
        return list(range(len(self.actions)))

    def get_next_states(self, state, action):
        p, state_next, _, _ = self.transitions[state][action][0]
        return {state_next: 1.0}

    def get_reward(self, state, action, state_next):
        _, _, reward, _ = self.transitions[state][action][0]
        return reward

    def get_transition_prob(self, state, action, state_next):
        p, next_state, _, _ = self.transitions[state][action][0]
        return p

    def is_terminal(self, state):
        done = self.transitions[state][0][0][3]
        return done

In [ ]:
taxi_mdp = TaxiMDPWrapper()
print("all_states (len) =", len(taxi_mdp.get_all_states()))
print("possible_actions(0) = ", taxi_mdp.get_possible_actions(0))
print("next_states(0, 0) = ", taxi_mdp.get_next_states(0, 0))
print("reward(0, 0, next) = ", taxi_mdp.get_reward(0, 0, list(taxi_mdp.get_next_states(0, 0).keys())[0]))
print("transition_prob(0, 0, next) = ",
      taxi_mdp.get_transition_prob(0, 0, list(taxi_mdp.get_next_states(0, 0).keys())[0]))

In [ ]:
GAMMA_TAXI = 0.99
state_values = value_iteration(taxi_mdp, gamma=GAMMA_TAXI, num_iter=200, min_difference=1e-5)

In [ ]:
from IPython.display import clear_output
from time import sleep

taxi_mdp_hist = TaxiMDPWrapper()
sv = {s : 0 for s in taxi_mdp_hist.get_all_states()}
history = []
for i in range(50):
    clear_output(True)
    print("after iteration %i"%i)
    sv = value_iteration(taxi_mdp_hist,
                            sv, gamma=GAMMA_TAXI, num_iter=1)
    history.append(sv[taxi_mdp_hist._initial_state])
    sleep(0.05)

## Задание 3. Демонстрация работы обученной стратегии

### Маршрут агента

In [ ]:
env = gym.make("Taxi-v3")
s = env.reset()
env.render()
trajectory = [s]
for t in range(100):
    a = get_optimal_action(taxi_mdp, state_values, s, GAMMA_TAXI)
    print(a, end='\n\n')
    s, r, done, _ = env.step(a)
    trajectory.append(s)
    env.render()
    if done:
        break
env.close()

In [ ]:
def taxi_pos_from_state(env_taxi, s):
    tr, tc, _, _ = env_taxi.unwrapped.decode(s)
    return tr, tc

def draw_taxi_route(trajectory, env_taxi):
    positions = [taxi_pos_from_state(env_taxi, s) for s in trajectory]
    rows = [p[0] for p in positions]
    cols = [p[1] for p in positions]

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.plot(cols, rows, 'b-o', markersize=6, linewidth=2, alpha=0.6)
    ax.plot(cols[0], rows[0], 'gs', markersize=15, label='Start')
    ax.plot(cols[-1], rows[-1], 'r*', markersize=20, label='End')

    locs = [(0, 0, 'R'), (0, 4, 'G'), (4, 0, 'Y'), (4, 3, 'B')]
    for r, c, name in locs:
        ax.annotate(name, (c, r), fontsize=14, fontweight='bold', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(4.5, -0.5)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.grid(True)
    ax.set_title("Маршрут таксиста")
    ax.legend()
    plt.show()

eplot = gym.make("Taxi-v3")
draw_taxi_route(trajectory, eplot)
eplot.close()

### Демонстрация на различных вариантах поля

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
e3 = gym.make("Taxi-v3")
for idx, ax in enumerate(axes):
    s = e3.reset()
    traj = [s]
    for t in range(200):
        a = get_optimal_action(taxi_mdp, state_values, s, GAMMA_TAXI)
        if a is None:
            break
        s, r, done, _ = e3.step(a)
        traj.append(s)
        if done:
            break
    positions = [taxi_pos_from_state(e3, st) for st in traj]
    rows = [p[0] for p in positions]
    cols = [p[1] for p in positions]
    ax.plot(cols, rows, 'b-o', markersize=5, linewidth=2, alpha=0.6)
    ax.plot(cols[0], rows[0], 'gs', markersize=12, label='Start')
    ax.plot(cols[-1], rows[-1], 'r*', markersize=18, label='End')
    locs = [(0, 0, 'R'), (0, 4, 'G'), (4, 0, 'Y'), (4, 3, 'B')]
    for r, c, name in locs:
        ax.annotate(name, (c, r), fontsize=12, fontweight='bold', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(4.5, -0.5)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.grid(True)
    ax.set_title(f"Эпизод {idx + 1}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
e3.close()

### Сходимость Value Function

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history, 'b-', linewidth=2)
plt.xlabel('Итерация')
plt.ylabel('V(s₀)')
plt.title('Сходимость Value Function по итерациям')
plt.grid(True)
plt.show()

### Среднее вознаграждение агента

In [ ]:
def generate_episode_optimal(env, episode_length=10**4):
    states, actions = [], []
    total_reward = 0.

    s = env.reset()

    for t in range(episode_length):

        a = get_optimal_action(taxi_mdp, state_values, s, GAMMA_TAXI)

        new_s, r, done, info = env.step(a)

        states.append(s)
        actions.append(a)
        total_reward += r

        s = new_s
        if done:
            break

    return states, actions, total_reward

num_episodes = 200
env_eval = gym.make("Taxi-v3")
sample_cumulative_rewards = [generate_episode_optimal(env_eval, episode_length=1000)[-1] for _ in range(num_episodes)]

plt.hist(sample_cumulative_rewards, bins=20)
plt.vlines([np.percentile(sample_cumulative_rewards, 50)], [0], [100], label="50'th percentile", color='green')
plt.vlines([np.percentile(sample_cumulative_rewards, 90)], [0], [100], label="90'th percentile", color='red')
plt.title("Taxi-v3 cumulative reward distribution")
plt.legend()
plt.show()
env_eval.close()